# Step 1: Train the model and create the app files
We train Isolation Forest on past transactions and save it as a file. Then we create `app.py`, the Streamlit app that loads that file.

**First:** drag `history.csv` into the Colab Files panel (the folder icon on the left).

## 1. Load the past transactions

In [1]:
import pandas as pd
import numpy as np
import joblib
import sklearn
from sklearn.ensemble import IsolationForest

history = pd.read_csv('history.csv')
history.head()

,transaction_id,amount,customer_typical_amount,transactions_last_hour,minutes_since_previous,account_age_days
0,TR-0000,59.47,47.38,2,52.844,692
1,TR-0001,76.92,80.28,2,768.572,1649
2,TR-0002,92.40,105.65,0,82.266,85
3,TR-0003,39.03,29.86,0,792.668,1320
4,TR-0004,112.16,120.12,1,44.043,529


## 2. Build the five inputs
The same five numbers the model saw in the Colab video. The app will build them in exactly the same way.

In [2]:
history['log_amount'] = np.log1p(history['amount'])
history['amount_ratio'] = history['amount'] / history['customer_typical_amount']
history['log_gap'] = np.log1p(history['minutes_since_previous'])
history['log_age'] = np.log1p(history['account_age_days'])

features = ['log_amount', 'amount_ratio', 'transactions_last_hour', 'log_gap', 'log_age']
X_history = history[features]
X_history.head()

,log_amount,amount_ratio,transactions_last_hour,log_gap,log_age
0,4.102147,1.255171,2,3.986091,6.541030
1,4.355683,0.958146,2,6.645835,7.408531
2,4.536891,0.874586,0,4.422040,4.454347
3,3.689629,1.307100,0,6.676665,7.186144
4,4.728803,0.933733,1,3.807618,6.272877


## 3. Train
200 trees of random cuts. No labels, just the five inputs.

In [3]:
model = IsolationForest(n_estimators=200, random_state=42)
model.fit(X_history)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


## 4. Score the past transactions
The app needs these scores to set its cutoff. For example, "send the top 4% to review" means: use the score that only 4% of past transactions are above.

In [4]:
history_scores = -model.score_samples(X_history)
print('Lowest score :', round(history_scores.min(), 3))
print('Highest score:', round(history_scores.max(), 3))
print('Cutoff for the top 4%:', round(np.percentile(history_scores, 96), 3))

Lowest score : 0.369
Highest score: 0.721
Cutoff for the top 4%: 0.566


## 5. Save the model
We save the model, the past scores, and the list of inputs in one file.

In [5]:
joblib.dump({'model': model, 'history_scores': history_scores, 'features': features}, 'model.joblib')

# a requirements file for Streamlit, using the same scikit-learn version as this notebook
with open('requirements.txt', 'w') as f:
    f.write('streamlit\npandas\nnumpy\njoblib\n')
    f.write(f'scikit-learn=={sklearn.__version__}\n')

print(open('requirements.txt').read())

streamlit
pandas
numpy
joblib
scikit-learn==1.9.1



## 6. Create the app
This cell writes the Streamlit app to a file called `app.py`. Run it as it is. You do not need to change anything. Read through the code if you want to see how the app uses the model.

In [6]:
%%writefile app.py
import time
import joblib
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title='Transaction Monitor', layout='wide')

# ---- load the trained model (made in the Colab notebook) ----
bundle = joblib.load('model.joblib')
model = bundle['model']
history_scores = bundle['history_scores']
features = bundle['features']


def score(amount, typical, last_hour, gap, age):
    """Turn one transaction into the five features and return its anomaly score."""
    row = pd.DataFrame([{
        'log_amount': np.log1p(amount),
        'amount_ratio': amount / typical,
        'transactions_last_hour': last_hour,
        'log_gap': np.log1p(gap),
        'log_age': np.log1p(age),
    }])[features]
    return -model.score_samples(row)[0]


# ---- sidebar: the review budget ----
st.title('Transaction Monitor')
st.write('Each new transaction is scored by Isolation Forest. '
         'If the score is above the cutoff, it goes to review.')

pct = st.sidebar.slider('Send this share of transactions to review (%)', 1, 10, 4)
cutoff = np.percentile(history_scores, 100 - pct)
st.sidebar.write(f'Score cutoff: **{cutoff:.3f}**')
st.sidebar.caption('The cutoff is set from past transactions: '
                   'a transaction scoring above it is more unusual than the top '
                   f'{pct}% of history.')

tab1, tab2 = st.tabs(['Check one transaction', 'Live stream'])

# ---- tab 1: type in one transaction ----
with tab1:
    c1, c2 = st.columns(2)
    amount = c1.number_input('Amount ($)', min_value=1.0, value=60.0)
    typical = c1.number_input("Customer's typical amount ($)", min_value=1.0, value=50.0)
    last_hour = c1.number_input('Transactions in the last hour', min_value=0, value=1)
    gap = c2.number_input('Minutes since previous transaction', min_value=0.0, value=120.0)
    age = c2.number_input('Account age (days)', min_value=0, value=700)

    if st.button('Score this transaction'):
        s = score(amount, typical, last_hour, gap, age)
        if s > cutoff:
            st.error(f'REVIEW  (score {s:.3f}, cutoff {cutoff:.3f})')
        else:
            st.success(f'OK  (score {s:.3f}, cutoff {cutoff:.3f})')
        share = (history_scores < s).mean() * 100
        st.write(f'This is more unusual than {share:.0f}% of past transactions.')

# ---- tab 2: transactions arriving one by one ----
with tab2:
    st.write('Press the button. The app receives the new transactions one at a time '
             'and scores each one as it arrives.')
    speed = st.select_slider('Speed', options=['Slow', 'Medium', 'Fast'], value='Medium')
    pause = {'Slow': 0.5, 'Medium': 0.15, 'Fast': 0.02}[speed]

    if st.button('Start the stream'):
        stream = pd.read_csv('new_transactions.csv')
        box = st.empty()
        counts = st.empty()
        st.subheader('Sent to review')
        table = st.empty()
        flagged = []

        for i, r in stream.iterrows():
            s = score(r['amount'], r['customer_typical_amount'],
                      r['transactions_last_hour'], r['minutes_since_previous'],
                      r['account_age_days'])
            msg = f"{r['transaction_id']}: ${r['amount']:.2f}  score {s:.3f}"
            if s > cutoff:
                flagged.append({'id': r['transaction_id'], 'amount': r['amount'],
                                'typical': r['customer_typical_amount'],
                                'last_hour': r['transactions_last_hour'],
                                'score': round(s, 3)})
                box.error(msg + '  ->  REVIEW')
            else:
                box.info(msg + '  ->  ok')
            counts.write(f'Received **{i + 1}** of {len(stream)}.  '
                         f'Sent to review: **{len(flagged)}**')
            if flagged:
                table.dataframe(pd.DataFrame(flagged), hide_index=True)
            time.sleep(pause)

        st.success(f'Done. {len(flagged)} transactions were sent to review.')

        # if the file has an answer key, check how we did
        if 'needs_review' in stream.columns:
            answer = stream.set_index('transaction_id')['needs_review']
            hits = sum(answer[f['id']] for f in flagged)
            st.write(f'{hits} of the {len(flagged)} flagged transactions really needed '
                     f'review. In total, {answer.sum()} transactions needed review.')

Writing app.py


## 7. Download the three files
You will upload them to GitHub in the next step.

In [ ]:
from google.colab import files
files.download('app.py')
files.download('model.joblib')
files.download('requirements.txt')